# Seasonal versus non-seasonal complete rolling-origin audit

This is the paired version of section 14 in `Outbreak_Probability_Run.ipynb`. It loads the independently fitted non-seasonal and seasonal Poisson parameter vectors. At every eligible historical Monday, both models see the same four-week history and forecast the same six verification weeks with matched random seeds. Each PDF page places the two complete fitted models side by side. Because both global vectors were fitted to the complete series, this is a conditional visual audit rather than a fully out-of-sample parameter-validation experiment.

In [ ]:
from pathlib import Path
import importlib
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
ROOT = Path.cwd().resolve()
if ROOT.name == 'outbreak_probability_model': ROOT = ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import outbreak_probability_model.london_calibration as london_calibration
london_calibration = importlib.reload(london_calibration)
import outbreak_probability_model.seasonal_rolling_audit as seasonal_rolling_audit
seasonal_rolling_audit = importlib.reload(seasonal_rolling_audit)
run_paired_seasonal_audit = seasonal_rolling_audit.run_paired_seasonal_audit
DEFAULT_LONDON_POISSON_FITTED_PARAMETERS = london_calibration.DEFAULT_LONDON_POISSON_FITTED_PARAMETERS
DEFAULT_LONDON_SEASONAL_POISSON_FITTED_PARAMETERS = london_calibration.DEFAULT_LONDON_SEASONAL_POISSON_FITTED_PARAMETERS
load_london_fitted_parameters = london_calibration.load_london_fitted_parameters
print('Loaded pipeline:', london_calibration.PIPELINE_VERSION)

## Controls

Run the non-seasonal and seasonal Poisson fitting notebooks first. This notebook reads both fitted vectors and the estimated seasonal amplitude and peak week automatically. The final setting uses 1,000 paths per model and origin; reduce `N_SIMULATIONS` only for a software check. `stride=1` evaluates every eligible weekly origin.

In [ ]:
for required_path, required_notebook in (
    (DEFAULT_LONDON_POISSON_FITTED_PARAMETERS,
     'London_Calibration_6Week_Rolling_Poisson_Fit.ipynb'),
    (DEFAULT_LONDON_SEASONAL_POISSON_FITTED_PARAMETERS,
     'London_Seasonal_Poisson_Fit.ipynb'),
):
    if not required_path.exists():
        raise FileNotFoundError(f'Run {required_notebook} first; missing {required_path}')
_, FITTED_SEASONAL_VECTOR = load_london_fitted_parameters(
    DEFAULT_LONDON_SEASONAL_POISSON_FITTED_PARAMETERS
)
SEASONAL_AMPLITUDE = FITTED_SEASONAL_VECTOR['seasonal_amplitude']
SEASONAL_PEAK_WEEK = FITTED_SEASONAL_VECTOR['seasonal_peak_week']
N_SIMULATIONS = 100
HORIZON_WEEKS = 6
OUTBREAK_THRESHOLD = 15.0
ALARM_PROBABILITY_CUTOFF = 0.40
STRIDE = 1
DISPLAY_ALL_ORIGIN_PAGES_INLINE = False  # Pages are still saved to the PDF and PNG folder.
print(f'Fitted seasonality: a={SEASONAL_AMPLITUDE:.4f}, peak week={SEASONAL_PEAK_WEEK:.3f}')

In [ ]:
OUTPUT = ROOT / 'experiments/measles/London/seasonal_complete_rolling_audit'
summary, pdf_path = run_paired_seasonal_audit(
    output_dir=OUTPUT,
    seasonal_amplitude=SEASONAL_AMPLITUDE,
    seasonal_peak_week=SEASONAL_PEAK_WEEK,
    n_simulations=N_SIMULATIONS,
    horizon_weeks=HORIZON_WEEKS,
    outbreak_threshold=OUTBREAK_THRESHOLD,
    alarm_probability_cutoff=ALARM_PROBABILITY_CUTOFF,
    stride=STRIDE,
    display_pages=DISPLAY_ALL_ORIGIN_PAGES_INLINE,
    nonseasonal_fitted_parameters_path=DEFAULT_LONDON_POISSON_FITTED_PARAMETERS,
    seasonal_fitted_parameters_path=DEFAULT_LONDON_SEASONAL_POISSON_FITTED_PARAMETERS,
)
display(summary)
print('Multipage paired audit:', pdf_path)
print('Individual origin PNGs:', OUTPUT / 'pages')
print('Origin-level metrics:', OUTPUT / 'paired_rolling_origin_metrics.csv')
print('Stochastic weekly paths:', OUTPUT / 'paired_rolling_stochastic_paths.csv')
print('Pooled calendar summary:', OUTPUT / 'paired_rolling_calendar_summary.csv')

## Pooled stochastic forecasts by target week

The band and median below are calculated directly from every stochastic path produced by the paired rolling audit. Predictions from all eligible origins and forecast horizons targeting the same calendar week are pooled. This is a rolling diagnostic: adjacent origins overlap and are not independent.

In [ ]:
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

calendar_summary = pd.read_csv(
    OUTPUT / 'paired_rolling_calendar_summary.csv',
    parse_dates=['target_date'],
)
stochastic_paths = pd.read_csv(
    OUTPUT / 'paired_rolling_stochastic_paths.csv',
    parse_dates=['origin_date', 'target_date'],
)

N_ORIGINS_TO_SHOW = 10
PATHS_PER_ORIGIN = 1
colours = {'non-seasonal': '#2878B5', 'seasonal': '#E1812C'}
model_order = ['non-seasonal', 'seasonal']
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True, sharey=True)

for ax, model_name in zip(axes, model_order):
    panel = (calendar_summary.loc[calendar_summary['model'].eq(model_name)]
             .sort_values('target_date'))
    dates = panel['target_date'].to_numpy()
    colour = colours[model_name]

    # Plot complete representative paths as separate six-week segments.
    model_paths = stochastic_paths.loc[stochastic_paths['model'].eq(model_name)]
    available_origins = np.sort(model_paths['origin_date'].unique())
    selected_positions = np.unique(np.linspace(
        0, len(available_origins) - 1,
        min(N_ORIGINS_TO_SHOW, len(available_origins)),
    ).round().astype(int))
    for origin_date in available_origins[selected_positions]:
        origin_paths = model_paths.loc[model_paths['origin_date'].eq(origin_date)]
        simulation_ids = np.sort(origin_paths['simulation'].unique())[:PATHS_PER_ORIGIN]
        for simulation_id in simulation_ids:
            segment = (origin_paths.loc[origin_paths['simulation'].eq(simulation_id)]
                       .sort_values('forecast_week'))
            path_dates = np.r_[np.datetime64(origin_date),
                               segment['target_date'].to_numpy(dtype='datetime64[ns]')]
            path_cases = np.r_[segment['origin_cases'].iloc[0],
                               segment['weekly_cases'].to_numpy(float)]
            ax.plot(path_dates, path_cases, color=colour, alpha=.20,
                    lw=.8, zorder=2)

    ax.fill_between(
        dates, panel['p10_cases'].to_numpy(float),
        panel['p90_cases'].to_numpy(float),
        color=colour, alpha=.18, linewidth=0, zorder=1,
    )
    ax.plot(
        dates, panel['median_cases'].to_numpy(float),
        color=colour, lw=2.3, zorder=3,
    )
    ax.plot(
        dates, panel['observed_cases'].to_numpy(float),
        color='#202020', marker='o', ms=3.2, lw=1.1, zorder=4,
    )
    ax.set_title(model_name.capitalize(), fontsize=13, fontweight='semibold', pad=10)
    ax.set_ylim(bottom=0)
    ax.margins(x=.01)
    ax.grid(axis='y', color='#D9D9D9', lw=.7, alpha=.65)
    ax.grid(axis='x', visible=False)
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle('Rolling six-week predictions versus confirmed cases',
             fontsize=16, fontweight='bold', y=.99)
fig.supxlabel('Week of symptom onset', fontsize=11, y=.04)
fig.supylabel('Cases per week', fontsize=11, x=.025)
fig.legend(
    handles=[
        Patch(facecolor='#6BAED6', alpha=.25, edgecolor='none',
              label='P10-P90 stochastic interval'),
        Line2D([0], [0], color='#777777', alpha=.45, lw=.8,
               label='Representative stochastic paths'),
        Line2D([0], [0], color='#555555', lw=2.3, label='Pooled stochastic median'),
        Line2D([0], [0], color='#202020', marker='o', ms=4, lw=1.1,
               label='Confirmed London cases'),
    ],
    loc='upper center', bbox_to_anchor=(.5, .925),
    ncol=4, frameon=False,
)
fig.tight_layout(rect=(.04, .07, 1, .86), w_pad=2.5)

rolling_figure = OUTPUT / 'rolling_predictions_vs_confirmed_cases.png'
fig.savefig(rolling_figure, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved pooled stochastic forecast figure:', rolling_figure)

## Confusion matrices

An alarm is issued when the forecast probability is at least `ALARM_PROBABILITY_CUTOFF`. Rows show what actually happened during the six-week outcome period and columns show the model decision. The seasonal and non-seasonal matrices use the same origins, observed outcomes and alarm threshold.

In [ ]:
audit_metrics = pd.read_csv(OUTPUT / 'paired_rolling_origin_metrics.csv')
audit_metrics['observed_event'] = audit_metrics['observed_event'].astype(bool)
audit_metrics['predicted_alarm'] = (
    audit_metrics['forecast_probability'] >= ALARM_PROBABILITY_CUTOFF
)

def safe_rate(numerator, denominator):
    return float(numerator / denominator) if denominator else np.nan

confusion_rows = []
confusion_matrices = {}
for model_name, group in audit_metrics.groupby('model', sort=False):
    actual = group['observed_event'].to_numpy(bool)
    predicted = group['predicted_alarm'].to_numpy(bool)
    tn = int((~actual & ~predicted).sum())
    fp = int((~actual & predicted).sum())
    fn = int((actual & ~predicted).sum())
    tp = int((actual & predicted).sum())
    matrix = np.array([[tn, fp], [fn, tp]], dtype=int)
    confusion_matrices[model_name] = matrix
    confusion_rows.append({
        'model': model_name, 'alarm_probability_cutoff': ALARM_PROBABILITY_CUTOFF,
        'true_negative': tn, 'false_positive': fp,
        'false_negative': fn, 'true_positive': tp,
        'sensitivity': safe_rate(tp, tp + fn),
        'specificity': safe_rate(tn, tn + fp),
        'precision': safe_rate(tp, tp + fp),
        'false_alarm_rate': safe_rate(fp, fp + tn),
        'miss_rate': safe_rate(fn, fn + tp),
        'accuracy': safe_rate(tp + tn, len(group)),
        'brier_score': float(group['brier_score'].mean()),
    })

confusion_metrics = pd.DataFrame(confusion_rows)
confusion_metrics.to_csv(OUTPUT / 'confusion_matrix_metrics.csv', index=False)
display(confusion_metrics)

model_order = [name for name in ('non-seasonal', 'seasonal') if name in confusion_matrices]
fig, axes = plt.subplots(1, len(model_order), figsize=(5.4 * len(model_order), 4.6))
axes = np.atleast_1d(axes)
maximum_count = max(matrix.max() for matrix in confusion_matrices.values())
for ax, model_name in zip(axes, model_order):
    matrix = confusion_matrices[model_name]
    image = ax.imshow(matrix, cmap='Blues', vmin=0, vmax=maximum_count or 1)
    for row in range(2):
        for column in range(2):
            value = matrix[row, column]
            colour = 'white' if value > 0.55 * (maximum_count or 1) else 'black'
            ax.text(column, row, str(value), ha='center', va='center',
                    fontsize=18, fontweight='bold', color=colour)
    ax.set_xticks([0, 1], ['No alarm', 'Alarm'])
    ax.set_yticks([0, 1], ['No outbreak', 'Outbreak'])
    ax.set_xlabel('Forecast decision')
    ax.set_ylabel('Observed outcome')
    ax.set_title(f"{model_name.capitalize()}\ncut-off = {ALARM_PROBABILITY_CUTOFF:.0%}")
fig.colorbar(image, ax=axes.tolist(), shrink=.78, label='Number of forecast origins')
fig.suptitle('Six-week outbreak confusion matrices', fontsize=14)
fig.subplots_adjust(left=.09, right=.90, bottom=.15, top=.80, wspace=.40)
confusion_figure = OUTPUT / 'seasonal_vs_nonseasonal_confusion_matrices.png'
fig.savefig(confusion_figure, dpi=180, bbox_inches='tight')
plt.show()
print('Saved confusion matrices:', confusion_figure)
print('Saved confusion metrics:', OUTPUT / 'confusion_matrix_metrics.csv')

## Forecast classifications across time

This overview marks every rolling forecast origin by its six-week outcome: green indicates a detected outbreak, red a missed outbreak, orange a false alarm, and blue a correctly quiet period. The detailed forecast-trajectory pages above use the same colours in their titles, borders and annotation boxes.

In [ ]:
classification_colours = {
    'DETECTED OUTBREAK': '#238b45',
    'MISSED OUTBREAK': '#c51b1d',
    'FALSE ALARM': '#d97706',
    'CORRECT BELOW THRESHOLD': '#4c78a8',
}
classification_order = [
    'DETECTED OUTBREAK', 'MISSED OUTBREAK',
    'FALSE ALARM', 'CORRECT BELOW THRESHOLD',
]
audit_metrics['origin_date'] = pd.to_datetime(audit_metrics['origin_date'])
model_order = [name for name in ('non-seasonal', 'seasonal')
               if name in set(audit_metrics['model'])]
fig, axes = plt.subplots(len(model_order), 1, figsize=(14, 7.5), sharex=True, sharey=True)
axes = np.atleast_1d(axes)
for ax, model_name in zip(axes, model_order):
    group = audit_metrics.loc[audit_metrics['model'].eq(model_name)].sort_values('origin_date')
    ax.plot(group['origin_date'], group['forecast_probability'], color='0.72', lw=1.0, zorder=1)
    ax.axhline(ALARM_PROBABILITY_CUTOFF, color='black', ls='--', lw=1.1,
               label=f'alarm cut-off ({ALARM_PROBABILITY_CUTOFF:.0%})')
    for classification in classification_order:
        selected = group.loc[group['classification'].eq(classification)]
        if selected.empty:
            continue
        ax.scatter(selected['origin_date'], selected['forecast_probability'],
                   s=38, color=classification_colours[classification],
                   edgecolor='white', linewidth=.45, label=classification, zorder=3)
    ax.set_title(model_name.capitalize(), loc='left', fontweight='bold')
    ax.set_ylabel('Forecast outbreak\nprobability')
    ax.set_ylim(-.03, 1.03)
    ax.grid(alpha=.20)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=5, frameon=False,
           bbox_to_anchor=(.5, .985))
axes[-1].set_xlabel('Forecast origin')
fig.suptitle('Classification of six-week outbreak forecasts', fontsize=15, y=1.035)
fig.tight_layout(rect=(0, 0, 1, .92))
classification_timeline = OUTPUT / 'seasonal_vs_nonseasonal_classification_timeline.png'
fig.savefig(classification_timeline, dpi=180, bbox_inches='tight')
plt.show()
display(
    audit_metrics.groupby(['model', 'classification'], as_index=False)
    .agg(origins=('origin_date', 'size'),
         mean_probability=('forecast_probability', 'mean'),
         mean_MAE=('median_mae', 'mean'))
)
print('Saved classification timeline:', classification_timeline)

## Reading the audit

For each displayed origin, weeks 1–6 are not used by the local four-week history-conditioning step. Compare whether the seasonal panel tracks rises and falls more closely, improves p10–p90 coverage, and reduces missed outbreaks without adding false alarms. Aggregate RMSE, MAE and Brier scores are saved separately. However, the global seasonal vector was estimated from the complete series, and adjacent origins overlap. Use this audit for visual diagnosis and paired score comparison; use a training-only frozen fit for the project's genuinely held-out performance claim.